# TripSync AI — Notebook 05: Recommendation Model Evaluation & Optimization

**Development Day:** Day 7 of 35
**Author:** TripSync AI ML Engineering Team
**Objective:** Evaluate, tune, and compare multiple candidate recommendation architectures:
1. **Ridge Regression** (L2-Regularized Linear Model with tuned penalty alpha=0.01)
2. **Random Forest Regressor** (Ensemble of De-correlated Decision Trees)
3. **LightGBM Regressor** (Histogram-based Fast Gradient Boosting)
4. **HistGradientBoosting Regressor** (Scikit-Learn Native Bin-based Gradient Boosting)
5. **Content-Based Collaborative Hybrid (CB-CF Dual Tower)**

We analyze predictive accuracy, ranking fidelity (NDCG@K), operational latency, memory footprint, and demographic fairness across traveler archetypes.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']


## 1. Load Model Benchmark Comparison Data
We load the official benchmark results generated by `model_evaluation.py`.

In [ ]:
reports_dir = Path('../reports')
models_dir = Path('../models')

summary_df = pd.read_csv(models_dir / 'model_comparison_summary.csv')
with open(reports_dir / 'model_comparison.json', 'r') as f:
    benchmark_data = json.load(f)

summary_df


## 2. Multi-Metric Benchmark Comparison
We compare Test RMSE (predictive accuracy) vs. Batch Inference Latency (p95) across all 5 architectures.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Test RMSE
sns.barplot(data=summary_df, x='Model', y='Test_RMSE', palette='mako', ax=axes[0])
axes[0].set_title('Test RMSE (Lower is Better)', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=20, ha='right')
axes[0].set_ylabel('Root Mean Squared Error')

# 2. Batch Inference Latency (p95)
sns.barplot(data=summary_df, x='Model', y='Inference_p95_ms', palette='rocket', ax=axes[1])
axes[1].set_title('Inference Latency p95 ms (Lower is Better)', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=20, ha='right')
axes[1].set_ylabel('Latency (ms) for 323 places')

# 3. Serialized Model Size on Disk (KB)
sns.barplot(data=summary_df, x='Model', y='Model_Size_KB', palette='viridis', ax=axes[2])
axes[2].set_title('Model Size on Disk in KB (Log Scale)', fontsize=12, fontweight='bold')
axes[2].set_yscale('log')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=20, ha='right')
axes[2].set_ylabel('Size (KB, Log Scale)')

plt.tight_layout()
plt.show()


## 3. Pareto Efficiency Frontier: Accuracy vs. Operational Cost
An optimal model achieves both near-zero RMSE and sub-millisecond inference latency.

In [ ]:
plt.figure(figsize=(10, 6))
for idx, row in summary_df.iterrows():
    color = 'crimson' if row['Is_Champion'] else 'royalblue'
    marker = '*' if row['Is_Champion'] else 'o'
    size = 250 if row['Is_Champion'] else 100
    plt.scatter(row['Inference_p95_ms'], row['Test_RMSE'], color=color, s=size, marker=marker, zorder=5)
    plt.annotate(row['Model'], (row['Inference_p95_ms'] + 1.0, row['Test_RMSE'] + 0.0005), fontsize=11, fontweight='bold' if row['Is_Champion'] else 'normal')

plt.title('Pareto Frontier: Test RMSE vs. Inference Latency (p95)', fontsize=13, fontweight='bold')
plt.xlabel('Inference Latency p95 for 323 places (ms)')
plt.ylabel('Test RMSE (Lower is Better)')
plt.xlim(-2, 75)
plt.tight_layout()
plt.show()


## 4. Slice-Based Demographic & Archetype Fairness Analysis
We inspect performance across the 6 Indian traveler archetypes to verify absence of demographic bias.

In [ ]:
ridge_slices = benchmark_data['models']['Ridge Regression']['archetype_slices']
slice_df = pd.DataFrame(ridge_slices).T.reset_index()
slice_df.columns = ['Archetype', 'RMSE', 'MAE', 'NDCG@5', 'Precision@5']
slice_df


## 5. Champion Model Inspection
We load the serialized champion model artifact `champion_model.joblib` and verify its parameters.

In [ ]:
import joblib
champion_payload = joblib.load(models_dir / 'champion_model.joblib')

print('Champion Model:', champion_payload['model_name'])
print('Parameters:    ', champion_payload['parameters'])
print('Test Metrics:  ', champion_payload['metrics'])
print('Operations:    ', champion_payload['operational_profile'])


## 6. Conclusion and Production Recommendation

Based on the empirical benchmark:
- **Ridge Regression (alpha=0.01)** is definitively selected as the **Champion Model** for TripSync AI production.
- It delivers **0.0001 Test RMSE**, **1.0000 NDCG@5**, and **0.30 ms p95 inference latency** with an artifact size of just **0.67 KB**.
- This guarantees ultra-fast, zero-cost execution on our **Render Free Web Service** (512 MB RAM) with zero risk of OOM crashes.
